# Modul 14: Studi Kasus 3 - Eksperimentasi A/B Testing & Optimasi Kinerja Sistem
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📌 1. Tujuan Pembelajaran
1. Merancang dan mengevaluasi eksperimen **A/B Testing** pada antarmuka sistem dan infrastruktur cloud.
2. Melakukan uji signifikansi komparatif (Two-Sample t-Test, Chi-Square Test of Proportions).
3. Mengambil kesimpulan rilis fitur perangkat lunak berdasarkan bukti statistik empiris.

---

## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Studi Kasus A/B Testing](images/img_14_case_abtesting_systems.png)

```
+------------------------------------------------------------------------------------+
|                   DESAIN EKSPERIMEN A/B TESTING SISTEM PERANGKAT LUNAK             |
+------------------------------------------------------------------------------------+
|                                                                                    |
|  [Trafik Pengguna (N=200)]                                                         |
|         |                                                                          |
|         +---> [Varian A - Kontrol (UI Lama)]   : Latency 3.4s, Conv 12%           |
|         |                                                                          |
|         +---> [Varian B - Optimasi (UI Baru)]  : Latency 1.9s, Conv 23%           |
|                                                                                    |
|  [Uji Statistik] : Independent t-Test (Latency) & Chi-Square (Conversion)          |
|  [Keputusan]     : p-value < 0.05 -> Rilis Varian B ke Seluruh Pengguna (Rollout)  |
+------------------------------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_ab = pd.read_csv("../datasets/10_ab_testing_system_metrics.csv")
print("Data Eksperimen A/B Testing dimuat. Total sesi:", len(df_ab))
display(df_ab.head())


## ⚡ 3. Pengujian Signifikansi Waktu Muat Halaman (*Page Load Time* - t-Test)


In [ ]:
group_a_time = df_ab[df_ab['variant_group'] == 'Variant_A_Control']['page_load_time_sec']
group_b_time = df_ab[df_ab['variant_group'] == 'Variant_B_Optimized']['page_load_time_sec']

t_stat, p_val_t = stats.ttest_ind(group_a_time, group_b_time, equal_var=False) # Welch t-test

print("=== Hasil Uji Komparasi Page Load Time (Welch t-Test) ===")
print(f"Rata-rata Varian A (Kontrol)  : {group_a_time.mean():.2f} detik (Std: {group_a_time.std():.2f})")
print(f"Rata-rata Varian B (Optimasi) : {group_b_time.mean():.2f} detik (Std: {group_b_time.std():.2f})")
print(f"Statistik t                   : {t_stat:.4f}")
print(f"p-value                       : {p_val_t:.4e}")
print(f">> Keputusan Statistik (α=0.05): {'Varian B Signifikan Lebih Cepat (H0 Ditolak)' if p_val_t < 0.05 else 'Tidak Ada Perbedaan Signifikan'}")


## 🛍️ 4. Pengujian Signifikansi Rasio Konversi (*Conversion Rate* - Chi-Square)


In [ ]:
contingency_ab = pd.crosstab(df_ab['variant_group'], df_ab['checkout_completed'])
chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_ab)

conv_rates = df_ab.groupby('variant_group')['checkout_completed'].mean() * 100

print("=== Hasil Uji Rasio Konversi Checkout (Chi-Square) ===")
print(f"Conversion Rate Varian A: {conv_rates['Variant_A_Control']:.1f}%")
print(f"Conversion Rate Varian B: {conv_rates['Variant_B_Optimized']:.1f}%")
print(f"Chi-Square Statistic    : {chi2_stat:.4f}, p-value: {p_val_chi2:.4e}")
print(f">> Keputusan Statistik (α=0.05): {'Peningkatan Konversi Signifikan Nyata (p < 0.05)' if p_val_chi2 < 0.05 else 'Perbedaan Tidak Signifikan'}")


## 📊 5. Visualisasi Hasil Eksperimen A/B Testing


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Distribusi Waktu Muat (Latency)
sns.kdeplot(data=df_ab, x='page_load_time_sec', hue='variant_group', fill=True, palette=['navy', 'coral'], ax=axes[0])
axes[0].set_title('Distribusi Waktu Muat Halaman: Varian A vs. Varian B', fontweight='bold')
axes[0].set_xlabel('Page Load Time (Detik)')
axes[0].set_ylabel('Kerapatan Densitas')

# Subplot 2: Perbandingan Rasio Konversi
conv_df = conv_rates.reset_index(name='Conversion Rate (%)')
sns.barplot(data=conv_df, x='variant_group', y='Conversion Rate (%)', palette=['navy', 'coral'], ax=axes[1])
axes[1].set_title(f'Perbandingan Rasio Konversi (Chi-Square p={p_val_chi2:.3f})', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for i, v in enumerate(conv_df['Conversion Rate (%)']):
    axes[1].text(i, v + 0.8, f"{v:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis & Rekomendasi Rilis

### Data Analysis Key Findings
* Varian B berhasil memangkas rata-rata waktu muat halaman dari **3.42 detik** menjadi **1.91 detik** dengan signifikansi statistik sangat tinggi ($t = 16.24$, $p < 0.001$).
* Rasio konversi checkout melonjak dari **12.0%** pada Varian A menjadi **23.0%** pada Varian B ($p = 0.041 < 0.05$).

### Actionable Product Insights
* Bukti statistik empiris merekomendasikan tim *Software Engineering* dan *Product Management* untuk melakukan **100% Full Rollout** antarmuka Varian B ke seluruh pengguna aktif.
